In [0]:

bootstrap_servers='pkc-xrnwx.asia-south2.gcp.confluent.cloud:9092'
api_key='ZZOYM2N4TCHR7PKB'
api_secret='cfltxH+Ioa6rAwNu5XGaKzPOelGXtA9F3Sypux4ZZSRM6upjVs6PSUulo1TV/ksg'
topic='credit_card_transactions'

In [0]:
import json
kafka_connection_json=dbutils.secrets.get(scope="fintech-scope",key="kafka_connection_details")
kafka_config=json.loads(kafka_connection_json)
bootstrap_servers=kafka_config['bootstrap_servers']
api_key=kafka_config['api_key']
api_secret=kafka_config['api_secret']
topic=kafka_config['topic']


In [0]:
jaas_config=f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'


In [0]:
sample_batch=(spark.read.format("kafka")
              .option("kafka.bootstrap.servers",bootstrap_servers)
              .option("subscribe",topic)
            .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", jaas_config)
        .option("startingOffsets","earliest") #here we have option to read all data from the beginning by default it is latest
        .load()
)

In [0]:
print(sample_batch.count())

In [0]:
display(sample_batch)

In [0]:
from pyspark.sql.functions import col
parsed_batch=sample_batch.select(
col("key").cast("string"),
col("value").cast("string"),
col("topic"),
col("partition"),
col("offset"),
col("timestamp"),
col("timestampType")
)

display(parsed_batch)

In [0]:
parsed_batch.write.saveAsTable("fintech.bronze.transactions_batch_test")

In [0]:
sample_batch_streaming=(spark.readStream.format("kafka")
              .option("kafka.bootstrap.servers",bootstrap_servers)
              .option("subscribe",topic)
            .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", jaas_config)
        .option("startingOffsets","earliest") #here we have option to read all data from the beginning by default it is latest
        .load()
)

#we cant display streaming data

In [0]:
from pyspark.sql.functions import col
parsed_streaming_df=sample_batch_streaming.select(
col("key").cast("string"),
col("value").cast("string"),
col("topic"),
col("partition"),
col("offset"),
col("timestamp"),
col("timestampType")
)


In [0]:
streaming_query=(parsed_streaming_df.writeStream.format("delta")
.outputMode("Append")
.option("checkpointLocation", "/Volumes/fintech/source/transaction/checkpoint/")
.trigger(availableNow=True) #here we can set trigger to read data in interval of time
.toTable("fintech.bronze.transactions_streaming_test")
)